# Enrichment - Map M Query Datasources

This notebook parses M queries from t_dataset_partitions, creates datasource lineage nodes, and maps datasource references to existing Fabric nodes (Lakehouse, Warehouse, SQL endpoint, Notebook, Dataflow).

In [ ]:
import time
import hashlib
import re
import pandas as pd
from pyspark.sql.types import StringType, StructField, StructType

_phase_timers = {}
_table_cache = {}

def start_phase(name):
    _phase_timers[name] = time.perf_counter()
    print(f"[timer] start {name}")

def end_phase(name):
    start = _phase_timers.get(name)
    if start is None:
        print(f"[timer] {name}: missing start")
        return
    elapsed = time.perf_counter() - start
    print(f"[timer] {name}: {elapsed:.2f}s")

def load_table(table_name):
    try:
        return spark.table(table_name).toPandas()
    except Exception:
        return pd.DataFrame()

def get_table(table_name):
    if table_name not in _table_cache:
        table_start = time.perf_counter()
        _table_cache[table_name] = load_table(table_name)
        print(f"[timer] load_table:{table_name}: {time.perf_counter() - table_start:.2f}s")
    return _table_cache[table_name]

def normalize_for_spark_write(df):
    if df is None:
        return pd.DataFrame()
    final_df = df.copy() if hasattr(df, 'copy') else pd.DataFrame(df)
    if final_df.empty and len(final_df.columns) == 0:
        return pd.DataFrame()
    for column_name in final_df.columns:
        final_df[column_name] = final_df[column_name].map(lambda value: None if pd.isna(value) or str(value).lower() == 'nan' else str(value))
    return final_df

def write_delta_table(df, table_name):
    final_df = normalize_for_spark_write(df)
    if final_df.empty and len(final_df.columns) == 0:
        return
    schema = StructType([StructField(column_name, StringType(), True) for column_name in final_df.columns])
    spark.createDataFrame(final_df, schema=schema).write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable(table_name)

StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can’t be run because you’ve hit a Spark compute or API rate limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.

In [ ]:
table_names = [
    't_dataset_partitions',
    't_dataset_tables',
    't_fabric_artifacts',
    't_lakehouse_metadata',
    't_warehouse_metadata',
    'v_nodes',
    'v_edges',
]

print('Registered tables for lazy loading:', ', '.join(table_names))
print('Tables will load on first use in lookup/mapping/merge phases.')

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
GUID_PATTERN = re.compile(r'[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[1-5][0-9a-fA-F]{3}-[89abAB][0-9a-fA-F]{3}-[0-9a-fA-F]{12}')
QUOTED_PATTERN = re.compile(r'"([^"\r\n]{3,})"|([^\r\n]{3,})')

def normalize_text(value):
    if value is None:
        return None
    text = str(value).strip()
    if not text or text.lower() == 'nan':
        return None
    return text

def classify_source_kind(query_text):
    text = (query_text or '').lower()
    if 'lakehouse' in text:
        return 'lakehouse'
    if 'warehouse' in text or 'datawarehouse' in text:
        return 'warehouse'
    if 'sql.database' in text or 'sql endpoint' in text or 'sqlendpoint' in text:
        return 'sql_endpoint'
    if 'dataflow' in text:
        return 'dataflow'
    if 'notebook' in text:
        return 'notebook'
    return 'datasource'

def extract_tokens(query_text):
    text = normalize_text(query_text) or ''
    ids = set([m.group(0).lower() for m in GUID_PATTERN.finditer(text)])
    quoted = set()
    for m in QUOTED_PATTERN.finditer(text):
        token = m.group(1) or m.group(2)
        token = normalize_text(token)
        if token:
            quoted.add(token.lower())
    return ids, quoted

def make_datasource_id(parts):
    joined = '|'.join([normalize_text(part) or '' for part in parts])
    digest = hashlib.sha1(joined.encode('utf-8')).hexdigest()[:20]
    return f'datasource:{digest}'

In [4]:
start_phase('build_artifact_lookup')

artifacts_table = get_table('t_fabric_artifacts')
artifacts = artifacts_table.copy() if not artifacts_table.empty else pd.DataFrame()
if artifacts.empty:
    artifacts = pd.DataFrame(columns=['id', 'display_name', 'name', 'type', 'workspace_id'])

for required_column in ['id', 'display_name', 'name', 'type', 'workspace_id']:
    if required_column not in artifacts.columns:
        artifacts[required_column] = None

artifacts['id_norm'] = artifacts['id'].map(lambda v: normalize_text(v).lower() if normalize_text(v) else None)
artifacts['type_norm'] = artifacts['type'].map(lambda v: normalize_text(v).lower() if normalize_text(v) else None)
artifacts['display_norm'] = artifacts['display_name'].map(lambda v: normalize_text(v).lower() if normalize_text(v) else None)
artifacts['name_norm'] = artifacts['name'].map(lambda v: normalize_text(v).lower() if normalize_text(v) else None)

lakehouse_meta_table = get_table('t_lakehouse_metadata')
warehouse_meta_table = get_table('t_warehouse_metadata')
lakehouse_meta = lakehouse_meta_table.copy() if not lakehouse_meta_table.empty else pd.DataFrame()
warehouse_meta = warehouse_meta_table.copy() if not warehouse_meta_table.empty else pd.DataFrame()

endpoint_by_artifact_id = {}
for meta_df in [lakehouse_meta, warehouse_meta]:
    if meta_df.empty:
        continue
    id_col = 'id' if 'id' in meta_df.columns else ('artifact_id' if 'artifact_id' in meta_df.columns else None)
    endpoint_col = None
    for candidate in ['sql_endpoint', 'sql_endpoint_url', 'sqlEndpoint', 'sqlEndpointUrl', 'connection_string', 'server']:
        if candidate in meta_df.columns:
            endpoint_col = candidate
            break
    if not id_col or not endpoint_col:
        continue
    for _, row in meta_df.iterrows():
        artifact_id = normalize_text(row.get(id_col))
        endpoint = normalize_text(row.get(endpoint_col))
        if artifact_id and endpoint:
            endpoint_by_artifact_id[artifact_id.lower()] = endpoint.lower()

artifact_node_by_id = {}
existing_nodes_table = get_table('v_nodes')
existing_nodes = existing_nodes_table.copy() if not existing_nodes_table.empty else pd.DataFrame(columns=['node_id', 'node_type'])
if not existing_nodes.empty:
    for _, row in existing_nodes.iterrows():
        node_id = normalize_text(row.get('node_id'))
        if node_id and node_id.startswith('artifact:'):
            artifact_id = node_id.split(':', 1)[1].lower()
            artifact_node_by_id[artifact_id] = node_id

print(f"Artifact lookup prepared: artifacts={len(artifacts)}, known_artifact_nodes={len(artifact_node_by_id)}, sql_endpoints={len(endpoint_by_artifact_id)}")
end_phase('build_artifact_lookup')

StatementMeta(, 59989136-12c0-49a0-adf4-83a74ebd6c9c, 8, Finished, Available, Finished, False)

In [16]:
TYPE_MATCH = {
    'lakehouse': {'lakehouse'},
    'warehouse': {'warehouse', 'mirroredwarehouse'},
    'sql_endpoint': {'lakehouse', 'warehouse', 'mirroredwarehouse', 'sqlendpoint'},
    'notebook': {'notebook'},
    'dataflow': {'dataflow', 'dataflowgen2'},
    'datasource': set(),
}

def find_artifact_match(query_text, source_kind):
    ids, tokens = extract_tokens(query_text)
    allowed_types = TYPE_MATCH.get(source_kind, set())

    candidates = artifacts
    if allowed_types:
        candidates = candidates[candidates['type_norm'].isin(allowed_types)]

    for artifact_id in ids:
        match = candidates[candidates['id_norm'] == artifact_id]
        if not match.empty:
            return match.iloc[0].to_dict(), 'id'

    for token in tokens:
        exact = candidates[(candidates['display_norm'] == token) | (candidates['name_norm'] == token)]
        if not exact.empty:
            return exact.iloc[0].to_dict(), 'name_exact'

    lowered_query = (query_text or '').lower()
    if source_kind == 'sql_endpoint':
        for _, artifact in candidates.iterrows():
            art_id = artifact.get('id_norm')
            endpoint = endpoint_by_artifact_id.get(art_id)
            if endpoint and endpoint in lowered_query:
                return artifact.to_dict(), 'sql_endpoint'

    return None, None

start_phase('map_m_datasources')
partitions_table = get_table('t_dataset_partitions')
partitions = partitions_table.copy() if not partitions_table.empty else pd.DataFrame()
if partitions.empty:
    print('No rows found in t_dataset_partitions. Nothing to enrich.')

for required_column in ['dataset_id', 'workspace_id', 'table_name', 'partition_name', 'query', 'source']:
    if required_column not in partitions.columns:
        partitions[required_column] = None

dataset_tables_table = get_table('t_dataset_tables')
dataset_tables = dataset_tables_table.copy() if not dataset_tables_table.empty else pd.DataFrame()
table_pk_lookup = {}
if not dataset_tables.empty:
    for _, row in dataset_tables.iterrows():
        dataset_id = normalize_text(row.get('dataset_id'))
        table_name = normalize_text(row.get('name') or row.get('table_name'))
        table_pk = normalize_text(row.get('table_pk'))
        if dataset_id and table_name and table_pk:
            table_pk_lookup[(dataset_id.lower(), table_name.lower())] = table_pk

new_nodes = []
new_edges = []
mapping_rows = []
seen_new_nodes = set()

processed_rows = 0
for _, row in partitions.iterrows():
    dataset_id = normalize_text(row.get('dataset_id'))
    workspace_id = normalize_text(row.get('workspace_id'))
    table_name = normalize_text(row.get('table_name'))
    partition_name = normalize_text(row.get('partition_name'))
    query_text = normalize_text(row.get('query')) or normalize_text(row.get('source')) or ''

    if not dataset_id or not table_name:
        continue

    source_kind = classify_source_kind(query_text)
    artifact_match, match_strategy = find_artifact_match(query_text, source_kind)

    table_pk = table_pk_lookup.get((dataset_id.lower(), table_name.lower()))
    if not table_pk:
        table_pk = f'{table_name}|{dataset_id}'
    from_node = f'{table_pk}'

    mapped_node_id = None
    datasource_node_type = source_kind
    datasource_display_name = None
    datasource_external_key = None

    if artifact_match:
        art_id = normalize_text(artifact_match.get('id'))
        art_type = normalize_text(artifact_match.get('type'))
        datasource_display_name = normalize_text(artifact_match.get('display_name')) or normalize_text(artifact_match.get('name')) or art_id
        datasource_external_key = art_id
        if art_id:
            mapped_node_id = artifact_node_by_id.get(art_id.lower(), art_id)
        datasource_node_type = normalize_text(art_type).lower() if normalize_text(art_type) else source_kind
    else:
        datasource_display_name = partition_name or table_name

    if mapped_node_id:
        to_node = mapped_node_id
        is_new_node = False
    else:
        to_node = make_datasource_id([dataset_id, table_name, partition_name, query_text[:400]])
        is_new_node = True

    if is_new_node and to_node not in seen_new_nodes:
        seen_new_nodes.add(to_node)
        new_nodes.append({
            'node_id': to_node,
            'parent_node': dataset_id,
            'node_name': datasource_display_name or 'Datasource',
            'dataset_id': dataset_id,
            'node_type': datasource_node_type or 'datasource',
            'workspace_id': workspace_id,
            'table_name': table_name
        })

    edge_id = make_datasource_id([from_node, to_node, partition_name or '', source_kind])
    new_edges.append({
        'edge_id': edge_id,
        'from_node': from_node,
        'to_node': to_node,
        'edge_type': f'm_query_source:{source_kind}',
        'dataset_id': dataset_id,
        'workspace_id': workspace_id
    })

    mapping_rows.append({
        'dataset_id': dataset_id,
        'workspace_id': workspace_id,
        'table_name': table_name,
        'partition_name': partition_name,
        'source_kind': source_kind,
        'mapped_node_id': to_node,
        'mapped_artifact_id': datasource_external_key,
        'match_strategy': match_strategy,
        'query_preview': query_text[:1000],
    })

    processed_rows += 1
    if processed_rows % 250 == 0:
        print(f"Processed partitions: {processed_rows}")

print(f"Mapped partition rows: {processed_rows}")
end_phase('map_m_datasources')

StatementMeta(, 59989136-12c0-49a0-adf4-83a74ebd6c9c, 57, Finished, Available, Finished, False)

In [17]:
start_phase('merge_and_write_outputs')
existing_nodes_table = get_table('v_nodes')
existing_edges_table = get_table('v_edges')
existing_nodes = existing_nodes_table.copy() if not existing_nodes_table.empty else pd.DataFrame(columns=['node_id', 'parent_node', 'node_name', 'dataset_id', 'node_type', 'workspace_id', 'table_name'])
existing_edges = existing_edges_table.copy() if not existing_edges_table.empty else pd.DataFrame(columns=['edge_id', 'from_node', 'to_node', 'edge_type', 'dataset_id', 'workspace_id'])

new_nodes_df = pd.DataFrame(new_nodes) if new_nodes else pd.DataFrame(columns=['node_id', 'parent_node', 'node_name', 'dataset_id', 'node_type', 'workspace_id', 'table_name'])
new_edges_df = pd.DataFrame(new_edges) if new_edges else pd.DataFrame(columns=['edge_id', 'from_node', 'to_node', 'edge_type', 'dataset_id', 'workspace_id'])
mapping_df = pd.DataFrame(mapping_rows) if mapping_rows else pd.DataFrame(columns=['dataset_id', 'workspace_id', 'table_name', 'partition_name', 'source_kind', 'mapped_node_id', 'mapped_artifact_id', 'match_strategy', 'query_preview'])

v_nodes_final = pd.concat([existing_nodes, new_nodes_df], axis=0, ignore_index=True)
if 'node_id' in v_nodes_final.columns:
    v_nodes_final = v_nodes_final.drop_duplicates(subset=['node_id'], keep='first')

v_edges_final = pd.concat([existing_edges, new_edges_df], axis=0, ignore_index=True)
if 'edge_id' in v_edges_final.columns:
    v_edges_final = v_edges_final.drop_duplicates(subset=['edge_id'], keep='first')

write_delta_table(v_nodes_final, 'v_nodes')
write_delta_table(v_edges_final, 'v_edges')
write_delta_table(mapping_df, 't_mquery_datasource_mappings')

print(f'Added {len(new_nodes_df)} datasource nodes (before dedupe)')
print(f'Added {len(new_edges_df)} datasource edges (before dedupe)')
print(f'Created t_mquery_datasource_mappings with {len(mapping_df)} rows')
print(f'v_nodes total rows after merge: {len(v_nodes_final)}')
print(f'v_edges total rows after merge: {len(v_edges_final)}')
end_phase('merge_and_write_outputs')

StatementMeta(, 59989136-12c0-49a0-adf4-83a74ebd6c9c, 58, Finished, Available, Finished, False)

Added 49 datasource nodes (before dedupe)
Added 49 datasource edges (before dedupe)
Created t_mquery_datasource_mappings with 49 rows
v_nodes total rows after merge: 545
v_edges total rows after merge: 1200


In [18]:
display(new_edges_df.head(50))

StatementMeta(, 59989136-12c0-49a0-adf4-83a74ebd6c9c, 59, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 047cfe6a-fb4b-49da-93eb-2250264c45f4)